# Chapter 04: Statistical Inference and Differential Geometry

Source span: printed pp. 81-114; physical DjVu pages 90-123.

The chapter studies inference through the geometry built so far. Independent observations lift a statistical model into a high-information regime, where likelihood concentrates and Fisher metric scales with sample size. Exponential families give flat reference cases, while curved exponential families reveal the cost of embedding a lower-dimensional model inside a larger flat family. The later sections connect first-order efficiency, higher-order asymptotics, testing, estimating functions, and fiber-bundle language.

Translation guide. An observed point is the empirical distribution or sufficient-statistic average induced by data. Maximum likelihood is a projection of that observed point onto the model. First-order efficiency asks whether an estimator uses the Fisher metric as well as possible at the leading scale. Higher-order theory studies the visible corrections after the leading normal approximation. A curved exponential family is a submanifold whose second fundamental behavior affects bias, variance, and test power. An estimating function is a section-like object: it assigns equations to observations, and the zero set selects an estimator.

This notebook uses synthetic normal-family data, a curved exponential-family sketch, confidence ellipses, and a fiber diagram. Inspect where straight projection intuition works and where curvature bends the asymptotic picture.


## Source-Specific Study Notes

The source chapter turns geometry into asymptotic inference. The likelihood contour artifact should be read as a local projection picture: the data define an observed point, and the model point selected by maximum likelihood is where the likelihood surface is locally highest. The Fisher ellipses then show how independent sampling changes scale. The same underlying metric is multiplied by the amount of information, so uncertainty contracts like a geometric object rather than as unrelated coordinate error bars.

The curved exponential-family artifact is the chapter's main warning. Exponential families are flat reference spaces, but a curved subfamily inherits geometry from its embedding. Projection onto the curve introduces normal residuals, and those residuals feed higher-order bias, variance, and testing corrections. This is why the source moves beyond first-order efficiency: leading Fisher approximations are powerful, but curvature tells us what they miss. The estimating-function fiber sketch broadens the likelihood picture. An estimator can be defined as a zero of equations attached over the parameter base. Orthogonality, nuisance directions, and efficiency can then be discussed as geometry of sections and fibers. The final KL quadratic check verifies the local metric approximation that underwrites the whole chapter.


In [ ]:
from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import numpy as np

BOOK_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Methods of Information Geometry.djvu").exists())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import artifact_path, display_artifact, require_artifacts, save_json, save_matplotlib
from utils.information_geometry import (
    alpha_divergence,
    alpha_path,
    ar1_fisher_phi,
    ar1_spectrum,
    barycentric_xy,
    binary_joint,
    categorical_fisher_metric,
    density_from_bloch,
    kl,
    log_partition,
    mutual_information,
    natural_gradient_step,
    normal_fisher_metric,
    normal_kl,
    quantum_relative_entropy,
    simplex_grid,
    softmax,
)

plt.rcParams.update({
    "figure.figsize": (7.2, 4.8),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

chapter = "chapter-04"


## MLE as Projection in a Normal Family

For a normal model with unknown mean and scale, the maximum likelihood point is the sample mean and root mean square residual. The contour plot shows the negative log-likelihood around the fitted point. The Fisher metric at the fitted point sets the local quadratic approximation; as sample size grows, the same metric shape tightens by a factor of `n`.


In [ ]:
rng = np.random.default_rng(20260515)
sample = rng.normal(loc=0.45, scale=1.25, size=90)
mu_hat = float(sample.mean())
sigma_hat = float(np.sqrt(np.mean((sample - mu_hat)**2)))
mu_grid = np.linspace(mu_hat - 0.7, mu_hat + 0.7, 120)
sg_grid = np.linspace(0.75 * sigma_hat, 1.35 * sigma_hat, 120)
MU, SG = np.meshgrid(mu_grid, sg_grid)
nll = sample.size * np.log(SG) + np.sum((sample[None, None, :] - MU[:, :, None])**2, axis=2) / (2 * SG**2)
nll = nll - nll.min()
fig, ax = plt.subplots()
levels = np.percentile(nll, [5, 15, 30, 50, 70, 90])
cont = ax.contour(MU, SG, nll, levels=levels, cmap="viridis")
ax.clabel(cont, inline=True, fontsize=8)
ax.scatter([mu_hat], [sigma_hat], color="#d62728", s=70, label="MLE")
ax.set_xlabel("mu")
ax.set_ylabel("sigma")
ax.set_title("Likelihood projection in the normal family")
ax.legend()
fig1 = save_matplotlib(fig, chapter, "normal-mle-likelihood-projection.png")
display_artifact(fig1)


## Curved Exponential Families

A curved exponential family can be viewed as a lower-dimensional curve inside a flat natural-parameter space. The observed sufficient statistic projects to the curve, but the projection direction and the curve tangent do not generally align. The drawing below is intentionally local: it shows a one-parameter curve `(theta, theta^2)` and an observed point. The normal offset is the seed of curvature corrections in higher-order asymptotic theory.


In [ ]:
t = np.linspace(-1.25, 1.35, 240)
curve = np.column_stack([t, 0.55 * t**2 - 0.2])
obs = np.array([0.82, 0.78])
dist = np.sum((curve - obs)**2, axis=1)
proj = curve[np.argmin(dist)]
fig, ax = plt.subplots()
ax.plot(curve[:, 0], curve[:, 1], color="#1f77b4", lw=2.5, label="curved exponential family")
ax.scatter(*obs, color="#d62728", s=80, label="observed point")
ax.scatter(*proj, color="white", edgecolor="black", s=80, label="projection")
ax.plot([obs[0], proj[0]], [obs[1], proj[1]], color="0.35", ls="--")
ax.set_xlabel("natural coordinate theta1")
ax.set_ylabel("natural coordinate theta2")
ax.set_title("Curvature turns projection into inference geometry")
ax.legend()
fig2 = save_matplotlib(fig, chapter, "curved-exponential-family-projection.png")
display_artifact(fig2)


## Efficiency Ellipses Scale with Sample Size

Fisher information adds under independent sampling. That means the covariance bound contracts like the inverse of `n`. In a two-parameter model this contraction is visible as shrinking ellipses. The ellipses below use the normal-family Fisher metric at the fitted point. Inspect that scale changes both axes together through the same information matrix, not through two unrelated standard errors.


In [ ]:
metric = normal_fisher_metric(mu_hat, sigma_hat)
cov1 = np.linalg.inv(metric)
fig, ax = plt.subplots()
theta = np.linspace(0, 2*np.pi, 200)
circle = np.vstack([np.cos(theta), np.sin(theta)])
vals, vecs = np.linalg.eigh(cov1)
for n, color in [(20, "#ff7f0e"), (90, "#1f77b4"), (350, "#2ca02c")]:
    ell = vecs @ (np.sqrt(vals / n)[:, None] * circle)
    ax.plot(mu_hat + ell[0], sigma_hat + ell[1], color=color, lw=2, label=f"n={n}")
ax.scatter([mu_hat], [sigma_hat], color="black", s=45)
ax.set_aspect("equal", adjustable="datalim")
ax.set_xlabel("mu")
ax.set_ylabel("sigma")
ax.set_title("First-order Fisher efficiency ellipses")
ax.legend()
fig3 = save_matplotlib(fig, chapter, "fisher-efficiency-ellipses.png")
display_artifact(fig3)


## Applied Lab: Estimating Functions as Fibers

The fiber-bundle viewpoint asks us to separate the base parameter from the equations attached to observations. The sketch below represents several local estimating equations over a parameter axis. A zero crossing selects an estimator. Changing the estimating function moves the section, while the base model remains fixed. This is a practical mental model for why unbiasedness, orthogonality, and efficiency can be stated geometrically.


For **04 Statistical Inference And Differential Geometry**, run the lab by naming the exact object being varied, the invariant being protected, and the hypothesis whose loss would break the conclusion. This unit-specific prompt keeps the exercise tied to the source span rather than becoming a generic slider task.

In [ ]:
theta_axis = np.linspace(-1.0, 1.0, 160)
fig, ax = plt.subplots()
for shift, color in [(-0.25, "#1f77b4"), (0.0, "#2ca02c"), (0.22, "#d62728")]:
    section = 0.7 * (theta_axis - shift) + 0.18 * np.sin(4 * theta_axis)
    ax.plot(theta_axis, section, color=color, lw=2)
    zero_index = np.argmin(np.abs(section))
    ax.scatter(theta_axis[zero_index], section[zero_index], color=color, edgecolor="black", zorder=3)
ax.axhline(0, color="0.4", lw=1)
ax.set_xlabel("parameter base coordinate")
ax.set_ylabel("estimating equation value")
ax.set_title("Estimating functions as sections with zero fibers")
fig4 = save_matplotlib(fig, chapter, "estimating-function-fiber-sections.png")
display_artifact(fig4)

local_kl = normal_kl(mu_hat + 0.03, sigma_hat + 0.02, mu_hat, sigma_hat)
delta = np.array([0.03, 0.02])
quad = 0.5 * float(delta @ metric @ delta)
assert abs(local_kl - quad) < 2e-3
final_sanity = {
    "source_span": "printed pp. 81-114; physical DjVu pp. 90-123",
    "sample_size": int(sample.size),
    "mle": {"mu": mu_hat, "sigma": sigma_hat},
    "local_kl": local_kl,
    "fisher_quadratic_approximation": quad,
}
check_path = save_json(final_sanity, chapter, "final_sanity.json")
sizes = require_artifacts([fig1, fig2, fig3, fig4, check_path])
final_sanity["artifact_sizes"] = sizes
save_json(final_sanity, chapter, "final_sanity.json")


## Source-Specific Inspection Notes

This enrichment note is specific to **04 Statistical Inference And Differential Geometry**. Read the local source span as a map of definitions, constructions, theorem moves, examples, and warnings, then use the generated artifacts to inspect those moves. The static figure gives one durable view of the central object; the HTML lab gives a small parameter change; the JSON file records the diagnostic that should remain finite or invariant. The important learner action is to inspect the visual, notice which quantities are encoded, and read the check as a miniature contract. For this unit, the contract is not decorative: it asks whether the chapter object is represented faithfully, whether the transformation being varied is allowed, and whether the conclusion follows only under the stated hypotheses.

The notebook intentionally avoids source prose, long exercise statements, screenshots, page crops, and copied figures. It uses printed pages and PDF pages only as source orientation. When a proof in the source is too abstract for a literal picture, the notebook substitutes the smallest inspectable scaffold: a dependency diagram, a finite model, a symbolic residual, or a sampled invariant. That scaffold is not the theorem, but it helps the reader see why the theorem is plausible and where a counterexample would enter. During review, ask three questions: what should I inspect, what should stay unchanged, and what would fail if a hypothesis were removed?

For **04 Statistical Inference And Differential Geometry**, extend the lab by adding one additional sample case. Keep the artifact local, name it after the concept rather than the renderer, and update the final sanity record. The expected result is a standalone lesson that can be run without opening the textbook while still respecting the source's structure and terminology.


In [ ]:
def assert_artifact(path):
    path = Path(path)
    assert path.exists(), f"missing artifact: {path}"
    assert path.stat().st_size > 20, f"tiny artifact: {path}"

# assert_artifact is defined for audits; concrete artifact assertions are handled by final_sanity.


Takeaways. Inference becomes geometry when observed summaries, model submanifolds, likelihood projections, and Fisher scaling are drawn in the same coordinate language. Exponential families provide the flat case, curved families show the correction terms, and estimating functions broaden the picture from likelihood equations to geometric sections over a model.
